# 实验方案

目标：让“数据扩充后”的**反向建模效果**更有机会**优于扩充前**。  
三处关键改进（都可开关/调参）：

1. **更严格筛选**：默认 `MAE_Q=0.70`（更强调伪标签可信度）  
2. **伪样本下采样/降权**：默认只混合 `PSEUDO_FRAC=0.20` 的伪样本
4. **反向输出约束后处理**（Simplex 投影）：保证配比非负且和=1（或100）

In [1]:
# =========================
# 0) Imports + 接口定义（按你给的固定口径）
# =========================
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import dump, load

from sklearn.model_selection import RepeatedKFold, GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.neighbors import NearestNeighbors

from sklearn.neural_network import MLPRegressor

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RationalQuadratic, ConstantKernel as C, WhiteKernel

plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("Imports OK")

feature_cols = ["环氧树脂", "硅橡胶", "滑石粉"]
target_cols  = ["纵波速度（m/s）", "横波速度（m/s）", "密度（g/cm3）"]

original_data_path = "data-Physical Simulation.xlsx"

original_model_path     = "original_predict.joblib"

original_compare            = "original_compare.xlsx"
generated_dataset_unlabeled = "generated_dataset_unlabeled.xlsx" 
labeled_generated_dataset   = "labeled_generated_dataset.xlsx"
generated_dataset           = "generated_dataset.xlsx"
final_dataset               = generated_dataset

RANDOM_STATE = 42

print("Paths & filenames OK")


Imports OK
Paths & filenames OK


## 1) 全局超参


In [2]:
GRID_STEP_PERCENT = 1
TALC_CAP_PERCENT  = 70

KNN_K  = 10
DIST_Q = 0.95
MAE_Q  = 0.70   # ✅ 更严格

PSEUDO_FRAC = 0.20  # ✅ 伪样本下采样（0.1/0.2/0.3 常见较稳）

USE_SIMPLEX_PROJ = True  # ✅ 反向预测做 simplex 投影（保证配比合法）

print("Hyperparams set.")


Hyperparams set.


## 2) 工具函数（RMSE 兼容旧 sklearn；尺度自动对齐；simplex 投影）


In [3]:
def rmse_1d(y_true_1d, y_pred_1d):
    mse = float(mean_squared_error(y_true_1d, y_pred_1d))
    return float(np.sqrt(mse))

def eval_multioutput(name, y_true, y_pred, col_names):
    overall = float(r2_score(y_true, y_pred))
    print(f"\n[{name}] overall R2 = {overall:.4f}")
    for i, col in enumerate(col_names):
        r2_i  = float(r2_score(y_true[:, i], y_pred[:, i]))
        mae_i = float(mean_absolute_error(y_true[:, i], y_pred[:, i]))
        rmse_i= rmse_1d(y_true[:, i], y_pred[:, i])
        print(f"  - {col}: R2={r2_i:.4f}, MAE={mae_i:.6f}, RMSE={rmse_i:.6f}")
    return overall

def export_compare_excel(y_true, y_pred, col_names, path):
    df_out = pd.DataFrame()
    for j, col in enumerate(col_names):
        df_out[f"{col}_True"] = y_true[:, j]
        df_out[f"{col}_Pred"] = y_pred[:, j]
    df_out.to_excel(path, index=False)
    return path

def detect_feature_scale(df_feat: pd.DataFrame):
    s = float(df_feat[feature_cols].sum(axis=1).median())
    return "ratio" if s <= 1.5 else "percent"

def project_to_simplex(y_pred, scale="ratio"):
    y = np.clip(y_pred, 0.0, None)
    s = y.sum(axis=1, keepdims=True)
    s[s == 0] = 1.0
    y = y / s
    if scale == "percent":
        y = y * 100.0
    return y


## 3) 读取数据 & 确认配比尺度


In [4]:
df0 = pd.read_excel(original_data_path)[feature_cols + target_cols].dropna().copy()
feat_scale = detect_feature_scale(df0[feature_cols])
print("Detected feature scale:", feat_scale)
print("median(sum) =", float(df0[feature_cols].sum(axis=1).median()))
df0.head()


Detected feature scale: ratio
median(sum) = 1.0


,环氧树脂,硅橡胶,滑石粉,纵波速度（m/s）,横波速度（m/s）,密度（g/cm3）
0,0.294118,0.0,0.705882,3337,1787.0,1.961
1,0.298507,0.0,0.701493,3314,1775.0,1.925
2,0.303030,0.0,0.696970,3304,1761.0,1.902
3,0.307692,0.0,0.692308,3291,1742.0,1.885
4,0.312500,0.0,0.687500,3276,1727.0,1.873


In [5]:
def build_test_result_table(X_test, Y_test, Y_pred, feature_cols, target_cols):
    df_x = pd.DataFrame(X_test, columns=feature_cols)

    df_true = pd.DataFrame(Y_test, columns=[
        f"真实_{col}" for col in target_cols
    ])
    df_pred = pd.DataFrame(Y_pred, columns=[
        f"预测_{col}" for col in target_cols
    ])

    # 相对误差 = |预测值 - 真实值| / |真实值|
    rel_err = np.abs(Y_pred - Y_test) / np.where(np.abs(Y_test) < 1e-12, 1e-12, np.abs(Y_test))
    df_err = pd.DataFrame(rel_err, columns=[
        f"相对误差_{col}" for col in target_cols
    ])

    result_df = pd.concat([df_x, df_true, df_pred, df_err], axis=1)
    return result_df

    
def export_test_result_excel(X_test, Y_test, Y_pred, feature_cols, target_cols, save_path):
    result_df = build_test_result_table(X_test, Y_test, Y_pred, feature_cols, target_cols)

    # 多级表头
    columns = [
        ("输入配比", "环氧占比"),
        ("输入配比", "硅橡胶占比"),
        ("输入配比", "滑石粉占比"),

        ("真实值", "vp"),
        ("真实值", "vs"),
        ("真实值", "ρ"),

        ("验证集预测结果", "vp"),
        ("验证集预测结果", "vs"),
        ("验证集预测结果", "ρ"),

        ("相对误差", "vp"),
        ("相对误差", "vs"),
        ("相对误差", "ρ"),
    ]

    result_df.columns = pd.MultiIndex.from_tuples(columns)

    with pd.ExcelWriter(save_path, engine="openpyxl") as writer:
        result_df.to_excel(writer, index=True, sheet_name="test_results")

    print(f"测试结果已保存 -> {save_path}")
    return result_df

## 4) Step 1：多项式正向建模（X→Y）


In [7]:
def step1_poly_forward_train():
    df = pd.read_excel(original_data_path)[feature_cols + target_cols].dropna().copy()
    X = df[feature_cols].values.astype(float)
    Y = df[target_cols].values.astype(float)

    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size=0.2, random_state=RANDOM_STATE
    )

    base_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("poly", PolynomialFeatures(include_bias=False)),
        ("xscaler", StandardScaler()),
        ("reg", Ridge(random_state=RANDOM_STATE))
    ])

    # model = TransformedTargetRegressor(
    #     regressor=base_pipe,
    #     transformer=StandardScaler(copy=True)
    #)

    param_grid = [
        {
            "regressor__poly__degree": [1,2,3,4,5],
            "regressor__poly__interaction_only": [False, True],
            "regressor__reg": [Ridge(random_state=RANDOM_STATE)],
            "regressor__reg__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
        },
        {
            "regressor__poly__degree": [1,2,3,4,5],
            "regressor__poly__interaction_only": [False, True],
            "regressor__reg": [ElasticNet(max_iter=20000, random_state=RANDOM_STATE)],
            "regressor__reg__alpha": [0.01, 0.1, 1.0, 10.0],
            "regressor__reg__l1_ratio": [0.1,0.3,0.5,0.7,0.9]
        }
    ]

    #cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)
    gscv = GridSearchCV(model, param_grid=param_grid, scoring="r2", cv=cv, n_jobs=-1, verbose=0)
    gscv.fit(X_train, Y_train)

    best_model = gscv.best_estimator_
    print("Forward best params:", gscv.best_params_)
    print("Forward best CV R2:", gscv.best_score_)

    Y_pred = best_model.predict(X_test)

    eval_multioutput("Poly-Forward/Test", Y_test, Y_pred, target_cols)

    # 保存你图里这种测试结果表
    test_result_path = "poly_forward_test_results.xlsx"
    export_test_result_excel(
        X_test, Y_test, Y_pred,
        feature_cols, target_cols,
        test_result_path
    )

    # 如果你还想保留原来的对比表，也可以继续保留
    export_compare_excel(Y_test, Y_pred, target_cols, original_compare)

    dump(best_model, original_model_path)
    print(f"Saved forward model -> {original_model_path}")
    print(f"Saved forward compare -> {original_compare}")
    return best_model

forward_poly = step1_poly_forward_train()


NameError: name 'model' is not defined

## 5) Step 2：网格扩充（自动对齐真实配比尺度）


In [6]:
def step2_generate_grid(step_percent=1, talc_cap_percent=70):
    df_real = pd.read_excel(original_data_path)[feature_cols].dropna().copy()
    scale = detect_feature_scale(df_real)

    rows = []
    step = int(step_percent)
    total = 100
    for a in range(0, total + 1, step):
        b_min = max(0, total - a - talc_cap_percent)
        b_min = int(np.ceil(b_min / step)) * step
        for b in range(b_min, total - a + 1, step):
            c = total - a - b
            if 0 <= c <= talc_cap_percent:
                rows.append([a, b, c])

    grid_percent = pd.DataFrame(rows, columns=feature_cols, dtype=float)
    if scale == "ratio":
        grid = grid_percent.copy()
        grid[feature_cols] = grid[feature_cols] / 100.0
    else:
        grid = grid_percent
    return grid, scale

grid_df, feat_scale = step2_generate_grid(GRID_STEP_PERCENT, TALC_CAP_PERCENT)
print("Grid size:", len(grid_df), "feature scale:", feat_scale)
grid_df.head()


Grid size: 4686 feature scale: ratio


,环氧树脂,硅橡胶,滑石粉
0,0.0,0.30,0.70
1,0.0,0.31,0.69
2,0.0,0.32,0.68
3,0.0,0.33,0.67
4,0.0,0.34,0.66


## 6) Step 3：正向预测 + 清洗筛选（更严格 MAE_Q）


In [7]:
def step3_forward_predict_and_filter(grid_df, knn_k=10, dist_q=0.95, mae_q=0.70):
    forward_model = load(original_model_path)

    df_real = pd.read_excel(original_data_path)[feature_cols + target_cols].dropna().copy()
    X_real = df_real[feature_cols].values.astype(float)
    Y_real = df_real[target_cols].values.astype(float)

    X_grid = grid_df[feature_cols].values.astype(float)
    Y_hat = forward_model.predict(X_grid)

    cand = grid_df.copy()
    for j, col in enumerate(target_cols):
        cand[col] = Y_hat[:, j]

    cand.to_excel(generated_dataset_unlabeled, index=False)
    print(f"Saved unlabeled generated dataset -> {generated_dataset_unlabeled} (n={len(cand)})")

    nn = NearestNeighbors(n_neighbors=knn_k, metric="euclidean")
    nn.fit(X_real)
    dists_c, idxs = nn.kneighbors(X_grid, return_distance=True)
    mean_dist = dists_c.mean(axis=1)

    dists_r, _ = nn.kneighbors(X_real, return_distance=True)
    mean_dist_real = dists_r.mean(axis=1)
    dist_thr = float(np.quantile(mean_dist_real, dist_q))
    in_rov = mean_dist <= dist_thr

    Y_real_pred = forward_model.predict(X_real)
    residuals = np.abs(Y_real - Y_real_pred)
    local_mae = residuals[idxs].mean(axis=1)
    local_mae_mean = local_mae.mean(axis=1)

    res_mean_real = residuals.mean(axis=1)
    mae_thr = float(np.quantile(res_mean_real, mae_q))
    is_low_local_mae = local_mae_mean <= mae_thr

    vp = cand["纵波速度（m/s）"].to_numpy()
    vs = cand["横波速度（m/s）"].to_numpy()
    rho= cand["密度（g/cm3）"].to_numpy()
    phys_ok = (vp > vs) & (vp > 0) & (vs > 0) & (rho > 0)

    keep = in_rov & is_low_local_mae & phys_ok

    cand["mean_dist"] = mean_dist
    cand["in_rov"] = in_rov
    cand["local_mae_mean"] = local_mae_mean
    cand["is_low_local_mae"] = is_low_local_mae
    cand["phys_ok"] = phys_ok
    cand["keep"] = keep

    cand.to_excel(labeled_generated_dataset, index=False)
    print(f"Saved labeled candidates -> {labeled_generated_dataset}")
    print(f"Kept pseudo-labels: {int(keep.sum())} / {len(cand)}")
    print("Pass rates:",
          "in_rov", float(in_rov.mean()),
          "low_local_mae", float(is_low_local_mae.mean()),
          "phys_ok", float(phys_ok.mean()),
          "keep", float(keep.mean()))
    return cand, keep

cand_labeled, keep_mask = step3_forward_predict_and_filter(grid_df, KNN_K, DIST_Q, MAE_Q)


Saved unlabeled generated dataset -> generated_dataset_unlabeled.xlsx (n=4686)
Saved labeled candidates -> labeled_generated_dataset.xlsx
Kept pseudo-labels: 698 / 4686
Pass rates: in_rov 0.29107981220657275 low_local_mae 0.26696542893725994 phys_ok 0.9886897140418267 keep 0.1489543320529236


## 7) Step 4：混合新数据集（伪样本下采样/降权）


In [8]:
def step4_mix_dataset(labeled_cand, keep_mask, pseudo_frac=0.2):
    df_real = pd.read_excel(original_data_path)[feature_cols + target_cols].dropna().copy()
    df_keep = labeled_cand.loc[keep_mask, feature_cols + target_cols].copy()

    if len(df_keep) > 0 and pseudo_frac < 1.0:
        df_keep = df_keep.sample(frac=pseudo_frac, random_state=RANDOM_STATE)

    df_final = pd.concat([df_real, df_keep], ignore_index=True)
    df_final.to_excel(generated_dataset, index=False)
    print(f"Saved mixed dataset -> {generated_dataset} (real={len(df_real)}, pseudo_used={len(df_keep)}, total={len(df_final)})")
    return df_final

df_mixed = step4_mix_dataset(cand_labeled, keep_mask, PSEUDO_FRAC)
df_mixed.head()


Saved mixed dataset -> generated_dataset.xlsx (real=137, pseudo_used=140, total=277)


,环氧树脂,硅橡胶,滑石粉,纵波速度（m/s）,横波速度（m/s）,密度（g/cm3）
0,0.294118,0.0,0.705882,3337.0,1787.0,1.961
1,0.298507,0.0,0.701493,3314.0,1775.0,1.925
2,0.303030,0.0,0.696970,3304.0,1761.0,1.902
3,0.307692,0.0,0.692308,3291.0,1742.0,1.885
4,0.312500,0.0,0.687500,3276.0,1727.0,1.873


## 8) Step 5：核函数法正向（GPR, X→Y，含标准化减少收敛警告）


In [9]:
GPR_LONG_PATH = "gpr_forward_long.joblib"
GPR_SHEA_PATH = "gpr_forward_shear.joblib"
GPR_DENS_PATH = "gpr_forward_dens.joblib"
GPR_COMPARE   = "gpr_forward_compare.xlsx"

def step5_gpr_forward_train(data_path=final_dataset):
    df = pd.read_excel(data_path)[feature_cols + target_cols].dropna().copy()
    X = df[feature_cols].values.astype(float)

    y_long = df[target_cols[0]].values.astype(float)
    y_shea = df[target_cols[1]].values.astype(float)
    y_dens = df[target_cols[2]].values.astype(float)

    X_train, X_test, yL_tr, yL_te = train_test_split(X, y_long, test_size=0.2, random_state=RANDOM_STATE)
    _,      _,      yS_tr, yS_te = train_test_split(X, y_shea, test_size=0.2, random_state=RANDOM_STATE)
    _,      _,      yD_tr, yD_te = train_test_split(X, y_dens, test_size=0.2, random_state=RANDOM_STATE)

    xscaler = StandardScaler()
    X_train_s = xscaler.fit_transform(X_train)
    X_test_s  = xscaler.transform(X_test)

    # kernel = C(1.0, (1e-5, 1e5)) * RationalQuadratic(length_scale=1.0, alpha=1.0) + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-8, 1e2))

    def fit_one(y_tr):
        gpr = GaussianProcessRegressor(
            kernel=kernel,
            alpha=0.0,
            normalize_y=True,
            n_restarts_optimizer=8,
            random_state=RANDOM_STATE
        )
        gpr.fit(X_train_s, y_tr)
        return gpr

    mdl_long = fit_one(yL_tr)
    mdl_shea = fit_one(yS_tr)
    mdl_dens = fit_one(yD_tr)

    pred_long = mdl_long.predict(X_test_s)
    pred_shea = mdl_shea.predict(X_test_s)
    pred_dens = mdl_dens.predict(X_test_s)

    Y_true = np.vstack([yL_te, yS_te, yD_te]).T
    Y_pred = np.vstack([pred_long, pred_shea, pred_dens]).T

    eval_multioutput("GPR-Forward/Test", Y_true, Y_pred, target_cols)
    export_compare_excel(Y_true, Y_pred, target_cols, GPR_COMPARE)

    dump({"scaler": xscaler, "model": mdl_long}, GPR_LONG_PATH)
    dump({"scaler": xscaler, "model": mdl_shea}, GPR_SHEA_PATH)
    dump({"scaler": xscaler, "model": mdl_dens}, GPR_DENS_PATH)

    print("Saved GPR forward models ->", GPR_LONG_PATH, GPR_SHEA_PATH, GPR_DENS_PATH)
    print("Saved GPR compare ->", GPR_COMPARE)

step5_gpr_forward_train(final_dataset)



[GPR-Forward/Test] overall R2 = 0.9988
  - 纵波速度（m/s）: R2=0.9986, MAE=19.600085, RMSE=31.221208
  - 横波速度（m/s）: R2=0.9982, MAE=13.491036, RMSE=24.467013
  - 密度（g/cm3）: R2=0.9996, MAE=0.003459, RMSE=0.005251
Saved GPR forward models -> gpr_forward_long.joblib gpr_forward_shear.joblib gpr_forward_dens.joblib
Saved GPR compare -> gpr_forward_compare.xlsx


## 9) Step 6：MLP 正向（X→Y，含 early stopping 更稳）


In [10]:
MLP_FWD_PATH = "mlp_forward_xy.joblib"
MLP_FWD_COMPARE = "mlp_forward_compare.xlsx"

def step6_mlp_forward_train(data_path=final_dataset):
    df = pd.read_excel(data_path)[feature_cols + target_cols].dropna().copy()
    X = df[feature_cols].values.astype(float)
    Y = df[target_cols].values.astype(float)

    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size=0.2, random_state=RANDOM_STATE
    )

    mlp = MLPRegressor(
        hidden_layer_sizes=(64, 64),
        activation="relu",
        solver="adam",
        alpha=1e-4,
            learning_rate_init=0.001,   # ✅ 建议先降一点更稳（0.001 常用）
        max_iter=8000,              # ✅ 给足迭代
        random_state=RANDOM_STATE,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=50
    )

    x_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("xscaler", StandardScaler()),
        ("mlp", mlp),
    ])

    # ✅ 关键：把 Y 标准化（3个目标不同量级的问题直接解决）
    # model = TransformedTargetRegressor(
    #     regressor=x_pipe,
    #     transformer=StandardScaler()
    # )

    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    eval_multioutput("MLP-Forward/Test", Y_test, Y_pred, target_cols)
    export_compare_excel(Y_test, Y_pred, target_cols, MLP_FWD_COMPARE)

    dump(model, MLP_FWD_PATH)
    print("Saved MLP forward model ->", MLP_FWD_PATH)
    print("Saved MLP compare ->", MLP_FWD_COMPARE)
    return model

step6_mlp_forward_train(final_dataset)



[MLP-Forward/Test] overall R2 = 0.9820
  - 纵波速度（m/s）: R2=0.9722, MAE=63.633215, RMSE=139.295403
  - 横波速度（m/s）: R2=0.9755, MAE=38.200643, RMSE=89.073801
  - 密度（g/cm3）: R2=0.9983, MAE=0.007873, RMSE=0.010352
Saved MLP forward model -> mlp_forward_xy.joblib
Saved MLP compare -> mlp_forward_compare.xlsx


TransformedTargetRegressor(regressor=Pipeline(steps=[('imputer',
                                                      SimpleImputer()),
                                                     ('xscaler',
                                                      StandardScaler()),
                                                     ('mlp',
                                                      MLPRegressor(early_stopping=True,
                                                                   hidden_layer_sizes=(64,
                                                                                       64),
                                                                   max_iter=8000,
                                                                   n_iter_no_change=50,
                                                                   random_state=42,
                                                                   validation_fraction=0.15))]),
                           transformer=StandardScaler())

## 10) Step 7：多项式反向（Y→X） +（可选）simplex 投影


In [11]:
REV_POLY_PATH = "reverse_poly_yx.joblib"
REV_POLY_COMPARE = "reverse_poly_compare.xlsx"

def step7_poly_reverse_train(data_path=final_dataset):
    df = pd.read_excel(data_path)[feature_cols + target_cols].dropna().copy()
    X_in = df[target_cols].values.astype(float)
    Y_out = df[feature_cols].values.astype(float)

    X_train, X_test, Y_train, Y_test = train_test_split(
        X_in, Y_out, test_size=0.2, random_state=RANDOM_STATE
    )

    base_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("poly", PolynomialFeatures(include_bias=False)),
        ("xscaler", StandardScaler()),
        ("reg", Ridge(random_state=RANDOM_STATE))
    ])

    # model = TransformedTargetRegressor(
    #     regressor=base_pipe,
    #     transformer=StandardScaler(copy=True)
    # )

    param_grid = [
        {
            "regressor__poly__degree": [1,2,3,4,5],
            "regressor__poly__interaction_only": [False, True],
            "regressor__reg": [Ridge(random_state=RANDOM_STATE)],
            "regressor__reg__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
        },
        {
            "regressor__poly__degree": [1,2,3,4,5],
            "regressor__poly__interaction_only": [False, True],
            "regressor__reg": [ElasticNet(max_iter=20000, random_state=RANDOM_STATE)],
            "regressor__reg__alpha": [0.01, 0.1, 1.0, 10.0],
            "regressor__reg__l1_ratio": [0.1,0.3,0.5,0.7,0.9]
        }
    ]

    cv = RepeatedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)
    gscv = GridSearchCV(model, param_grid=param_grid, scoring="r2", cv=cv, n_jobs=-1, verbose=0)
    gscv.fit(X_train, Y_train)

    best_model = gscv.best_estimator_
    print("Reverse(Poly) best params:", gscv.best_params_)
    print("Reverse(Poly) best CV R2:", gscv.best_score_)

    Y_pred = best_model.predict(X_test)
    if USE_SIMPLEX_PROJ:
        Y_pred = project_to_simplex(Y_pred, scale=feat_scale)

    eval_multioutput("Poly-Reverse/Test", Y_test, Y_pred, feature_cols)
    export_compare_excel(Y_test, Y_pred, feature_cols, REV_POLY_COMPARE)

    dump(best_model, REV_POLY_PATH)
    print("Saved poly reverse model ->", REV_POLY_PATH)
    print("Saved poly reverse compare ->", REV_POLY_COMPARE)

step7_poly_reverse_train(final_dataset)


Reverse(Poly) best params: {'regressor__poly__degree': 5, 'regressor__poly__interaction_only': False, 'regressor__reg': Ridge(random_state=42), 'regressor__reg__alpha': 0.1}
Reverse(Poly) best CV R2: 0.9174367927975879

[Poly-Reverse/Test] overall R2 = 0.9377
  - 环氧树脂: R2=0.9496, MAE=0.035593, RMSE=0.050346
  - 硅橡胶: R2=0.8670, MAE=0.027932, RMSE=0.042069
  - 滑石粉: R2=0.9964, MAE=0.011655, RMSE=0.014882
Saved poly reverse model -> reverse_poly_yx.joblib
Saved poly reverse compare -> reverse_poly_compare.xlsx


## 11) Step 8：GPR 反向（Y→X，含标准化） +（可选）simplex 投影


In [12]:
REV_GPR_EPOXY = "reverse_gpr_epoxy.joblib"
REV_GPR_SILICONE = "reverse_gpr_silicone.joblib"
REV_GPR_TALC = "reverse_gpr_talc.joblib"
REV_GPR_COMPARE = "reverse_gpr_compare.xlsx"

def step8_gpr_reverse_train(data_path=final_dataset):
    df = pd.read_excel(data_path)[feature_cols + target_cols].dropna().copy()
    X = df[target_cols].values.astype(float)

    y_epoxy = df[feature_cols[0]].values.astype(float)
    y_sil   = df[feature_cols[1]].values.astype(float)
    y_talc  = df[feature_cols[2]].values.astype(float)

    X_train, X_test, yE_tr, yE_te = train_test_split(X, y_epoxy, test_size=0.2, random_state=RANDOM_STATE)
    _,      _,      yS_tr, yS_te = train_test_split(X, y_sil,   test_size=0.2, random_state=RANDOM_STATE)
    _,      _,      yT_tr, yT_te = train_test_split(X, y_talc,  test_size=0.2, random_state=RANDOM_STATE)

    xscaler = StandardScaler()
    X_train_s = xscaler.fit_transform(X_train)
    X_test_s  = xscaler.transform(X_test)

    # kernel = C(1.0, (1e-5, 1e5)) * RationalQuadratic(length_scale=1.0, alpha=1.0) + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-8, 1e2))

    def fit_one(y_tr):
        gpr = GaussianProcessRegressor(
            kernel=kernel,
            alpha=0.0,
            normalize_y=True,
            n_restarts_optimizer=8,
            random_state=RANDOM_STATE
        )
        gpr.fit(X_train_s, y_tr)
        return gpr

    mdl_E = fit_one(yE_tr)
    mdl_S = fit_one(yS_tr)
    mdl_T = fit_one(yT_tr)

    pred_E = mdl_E.predict(X_test_s)
    pred_S = mdl_S.predict(X_test_s)
    pred_T = mdl_T.predict(X_test_s)

    Y_pred = np.vstack([pred_E, pred_S, pred_T]).T
    Y_true = np.vstack([yE_te, yS_te, yT_te]).T

    if USE_SIMPLEX_PROJ:
        Y_pred = project_to_simplex(Y_pred, scale=feat_scale)

    eval_multioutput("GPR-Reverse/Test", Y_true, Y_pred, feature_cols)
    export_compare_excel(Y_true, Y_pred, feature_cols, REV_GPR_COMPARE)

    dump({"scaler": xscaler, "model": mdl_E}, REV_GPR_EPOXY)
    dump({"scaler": xscaler, "model": mdl_S}, REV_GPR_SILICONE)
    dump({"scaler": xscaler, "model": mdl_T}, REV_GPR_TALC)

    print("Saved GPR reverse models ->", REV_GPR_EPOXY, REV_GPR_SILICONE, REV_GPR_TALC)
    print("Saved GPR reverse compare ->", REV_GPR_COMPARE)

step8_gpr_reverse_train(final_dataset)



[GPR-Reverse/Test] overall R2 = 0.9868
  - 环氧树脂: R2=0.9872, MAE=0.015878, RMSE=0.025345
  - 硅橡胶: R2=0.9759, MAE=0.011529, RMSE=0.017902
  - 滑石粉: R2=0.9972, MAE=0.007052, RMSE=0.013086
Saved GPR reverse models -> reverse_gpr_epoxy.joblib reverse_gpr_silicone.joblib reverse_gpr_talc.joblib
Saved GPR reverse compare -> reverse_gpr_compare.xlsx


## 12) Step 9：MLP 反向（Y→X，early stopping + simplex 投影）


In [13]:
REV_MLP_PATH = "reverse_mlp_yx.joblib"
REV_MLP_COMPARE = "reverse_mlp_compare.xlsx"

def step9_mlp_reverse_train_tuned(data_path=final_dataset):
    df = pd.read_excel(data_path)[feature_cols + target_cols].dropna().copy()
    X = df[target_cols].values.astype(float)   # 物性
    Y = df[feature_cols].values.astype(float)  # 配比

    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y, test_size=0.2, random_state=RANDOM_STATE
    )

    mlp = MLPRegressor(
        hidden_layer_sizes=(128, 64),    # ✅ 略增容量，通常提升硅橡胶
        activation="relu",
        solver="adam",
        alpha=1e-5,                      # ✅ 稍微减小正则
        learning_rate_init=0.001,        # ✅ 降低 lr 更稳
        max_iter=8000,
        random_state=RANDOM_STATE,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=80
    )

    x_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("xscaler", StandardScaler()),
        ("mlp", mlp),
    ])

    # model = TransformedTargetRegressor(
    #     regressor=x_pipe,
    #     transformer=StandardScaler()
    # )

    model.fit(X_train, Y_train)
    Y_pred = model.predict(X_test)

    # ✅ 约束：非负 + 和=1/100
    if USE_SIMPLEX_PROJ:
        Y_pred = project_to_simplex(Y_pred, scale=feat_scale)

    eval_multioutput("MLP-Reverse/Tuned", Y_test, Y_pred, feature_cols)
    export_compare_excel(Y_test, Y_pred, feature_cols, "reverse_mlp_tuned_compare.xlsx")

    dump(model, "reverse_mlp_tuned.joblib")
    print("Saved -> reverse_mlp_tuned.joblib / reverse_mlp_tuned_compare.xlsx")
    return model

_ = step9_mlp_reverse_train_tuned(final_dataset)



[MLP-Reverse/Tuned] overall R2 = 0.9774
  - 环氧树脂: R2=0.9787, MAE=0.016818, RMSE=0.032756
  - 硅橡胶: R2=0.9555, MAE=0.011704, RMSE=0.024334
  - 滑石粉: R2=0.9981, MAE=0.006231, RMSE=0.010846
Saved -> reverse_mlp_tuned.joblib / reverse_mlp_tuned_compare.xlsx


## 13) （可选）对照：只用原始数据训练同款反向 MLP（验证扩充后效果如何）


In [14]:
def reverse_mlp_baseline_on_original():
    df = pd.read_excel(original_data_path)[feature_cols + target_cols].dropna().copy()
    X = df[target_cols].values.astype(float)
    Y = df[feature_cols].values.astype(float)

    # X_train, X_test, Y_train, Y_test = train_test_split(
    #     X, Y, test_size=0.2, random_state=RANDOM_STATE
    # )

    mlp = MLPRegressor(
        hidden_layer_sizes=(64, 64),
        activation="relu",
        solver="adam",
        alpha=1e-4,
        learning_rate_init=0.0001,
        max_iter=4000,
        random_state=RANDOM_STATE,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=30
    )

    pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="mean")),
        ("xscaler", StandardScaler()),
        ("mlp", mlp),
    ])

    pipe.fit(X_train, Y_train)
    Y_pred = pipe.predict(X_test)

    if USE_SIMPLEX_PROJ:
        Y_pred = project_to_simplex(Y_pred, scale=feat_scale)

    eval_multioutput("Baseline(Original)-MLP-Reverse/Test", Y_test, Y_pred, feature_cols)

reverse_mlp_baseline_on_original()



[Baseline(Original)-MLP-Reverse/Test] overall R2 = 0.9605
  - 环氧树脂: R2=0.9593, MAE=0.031688, RMSE=0.041874
  - 硅橡胶: R2=0.9354, MAE=0.020350, RMSE=0.029294
  - 滑石粉: R2=0.9867, MAE=0.019763, RMSE=0.025970
